# MNIST Persistent Laplacian Experiment
Code based on: *The Persistent Laplacian for Data Science* (Davies, Wan, Sanchez-Garcia, ICML 2023)

## 0. Imports

In [ ]:
import sys

REPO_ROOT = '../'
sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import random
import warnings
from datetime import datetime
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from gtda.images import Binarizer, HeightFiltration

from src.utils import cubical_complex
from src.perslap import compute_pers_lap_pair

warnings.filterwarnings('ignore')
print('Imports fine')

## 1. Load Data

In [ ]:
print('Loading MNIST...')
X_full, y_full = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
X_full = X_full.reshape((-1, 28, 28))
print(f'Loaded: {X_full.shape[0]} images')

## 2. Feature Functions

In [ ]:
RESOLUTION = 5
DIRECTION  = [1, 0]


def compute_raw_spectra(image, resolution=5, direction=None):
    """Computes the persistent Laplacian spectra for a single image."""
    if direction is None:
        direction = [1, 0]
    binarizer         = Binarizer(threshold=0.4)
    filtration_fitter = HeightFiltration(direction=np.array(direction))
    image_binarised   = binarizer.fit_transform(image[None, :, :])
    filtration        = filtration_fitter.fit_transform(image_binarised)

    max_val   = np.max(filtration)
    min_val   = np.min(filtration)
    increment = (max_val - min_val) / resolution
    vals      = [min_val + i * increment for i in range(resolution)]
    vals[-1]  = max_val

    all_spectra = []
    for i, k in enumerate(vals):
        K = cubical_complex(filtration, k)
        for l in vals[i:]:
            L = cubical_complex(filtration, l)
            _, spectra = compute_pers_lap_pair(K, L, verbose=False, complex_type='cubical')
            all_spectra.append([np.real(s) for s in spectra])
    return all_spectra


# --- Auxiliar function: sorts eigenvalues ---
def _sorted_pos(spec_dim, k=2):
    """Returns the k smallest positive eigenvalues of a given spectrum, padded with zeros if necessary."""
    rounded = np.round(spec_dim, 3)
    nonzero = np.sort(rounded[np.nonzero(rounded)])
    result  = np.zeros(k)
    result[:min(k, len(nonzero))] = nonzero[:k]
    return result


# --- Five Feature Functions ---

def feats_A(all_spectra):
    """(A) Smallest positive eigenvalue."""
    vec = []
    for spectra in all_spectra:
        for spec_dim in spectra:
            vec.append(_sorted_pos(spec_dim, k=1)[0])
    return np.array(vec)


def feats_B(all_spectra):
    """(B) Number of zero eigenvalues."""
    vec = []
    for spectra in all_spectra:
        for spec_dim in spectra:
            rounded = np.round(spec_dim, 3)
            vec.append(float(np.sum(rounded == 0.0)))
    return np.array(vec)


def feats_C(all_spectra):
    """(C) Number of zeros + smallest positive eigenvalue."""
    vec = []
    for spectra in all_spectra:
        for spec_dim in spectra:
            rounded  = np.round(spec_dim, 3)
            n_zeros  = float(np.sum(rounded == 0.0))
            min1     = _sorted_pos(spec_dim, k=1)[0]
            vec.extend([n_zeros, min1])
    return np.array(vec)


def feats_D(all_spectra):
    """(D) Smallest and second smallest positive eigenvalues."""
    vec = []
    for spectra in all_spectra:
        for spec_dim in spectra:
            top2 = _sorted_pos(spec_dim, k=2)
            vec.extend([top2[0], top2[1]])
    return np.array(vec)


def feats_E(all_spectra):
    """(E) Number of zeros + smallest two positive eigenvalues."""
    vec = []
    for spectra in all_spectra:
        for spec_dim in spectra:
            rounded = np.round(spec_dim, 3)
            n_zeros = float(np.sum(rounded == 0.0))
            top2    = _sorted_pos(spec_dim, k=2)
            vec.extend([n_zeros, top2[0], top2[1]])
    return np.array(vec)


FEAT_FUNCS = {
    '(A) min ev':              feats_A,
    '(B) number zeroes':       feats_B,
    '(C) zeroes + min ev':     feats_C,
    '(D) min + 2min ev':       feats_D,
    '(E) zeroes + min + 2min': feats_E,
}


def pad(list_of_vecs):
    max_len = max(len(v) for v in list_of_vecs)
    return np.array([np.pad(v, (0, max_len - len(v))) for v in list_of_vecs])

## 3. Experiment

Per Run:
1. Randomly draw N_SAMPLES images
2. Compute Spectra 
3. Compute five different Features
4. Train SVM trainieren and measure accuracy (80/20 split)

The N_RUNS runs are done in parallel, on N_JOBS CPU-Kernen.

> **Run time:** with N_SAMPLES=2000 it takes ~11 min per run

In [ ]:
N_RUNS     = 10    # number of repetitions
N_SAMPLES  = 2000  # images per run
N_JOBS     = 10    # parallel processes
TEST_SIZE  = 0.2


def run_single_experiment(run_id, X_full, y_full, n_samples, resolution, direction):
    """
    A complete experiment:
    - draws n_samples random images
    - computes spectra
    - returns dict {variant: accuracy}
    """
    rng = np.random.RandomState(run_id)  # reproducible but different for each run
    idx = rng.choice(len(X_full), size=n_samples, replace=False)
    X   = X_full[idx]
    y   = y_full[idx]

    # compute spectra
    raw_all = []
    failed  = 0
    for image in X:
        try:
            raw_all.append(compute_raw_spectra(image, resolution=resolution, direction=direction))
        except Exception:
            raw_all.append(None)
            failed += 1

    valid_mask = [r is not None for r in raw_all]
    raw_valid  = [r for r in raw_all if r is not None]
    y_valid    = y[valid_mask]

    accs = {}
    for name, fn in FEAT_FUNCS.items():
        X_feat = pad([fn(r) for r in raw_valid])

        X_train, X_test, y_train, y_test = train_test_split(
            X_feat, y_valid, test_size=TEST_SIZE,
            random_state=run_id, stratify=y_valid
        )
        scaler = StandardScaler()
        clf    = SVC(kernel='rbf', C=1.0, random_state=run_id)
        clf.fit(scaler.fit_transform(X_train), y_train)
        accs[name] = accuracy_score(y_test, clf.predict(scaler.transform(X_test)))

    print(f'  Run {run_id:2d} done ({failed} errors)')
    return accs


t0 = datetime.now()

all_results = Parallel(n_jobs=N_JOBS, verbose=0)(
    delayed(run_single_experiment)(run_id, X_full, y_full, N_SAMPLES, RESOLUTION, DIRECTION)
    for run_id in range(N_RUNS)
)

elapsed = (datetime.now() - t0).seconds
print(f'\nDone in {elapsed}s ({elapsed//60}min {elapsed%60}s)')

## 4. Evaluation: Mean and Standard Deviation

In [ ]:
results_df = pd.DataFrame(all_results)  # shape: (N_RUNS, 5)

summary = pd.DataFrame({
    'mean': results_df.mean(),
    'std':        results_df.std(),
    'min':        results_df.min(),
    'max':        results_df.max(),
})
summary = summary.sort_values('mean', ascending=False)

print(summary.to_string(float_format='{:.4f}'.format))

In [ ]:
# bar plot with error bars
fig, ax = plt.subplots(figsize=(10, 5))

names  = summary.index.tolist()
means  = summary['mean'].values
stds   = summary['std'].values
colors = ['steelblue', 'darkorange', 'seagreen', 'mediumpurple', 'crimson']

bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=colors[:len(names)], alpha=0.85, error_kw={'linewidth': 2})

ax.set_ylim(0, min(1.0, means.max() + stds.max() + 0.1))
ax.set_ylabel('Accuracy')
ax.set_title(f'Comparison\n(SVM rbf, {N_RUNS} Runs x {N_SAMPLES} Images, mean +/- std)')
ax.tick_params(axis='x', labelsize=9)

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.005,
            f'{m:.3f}\n±{s:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150)
plt.show()
print('Saved: results_comparison.png')

In [ ]:
# Boxplot of all individual runs
fig, ax = plt.subplots(figsize=(10, 5))
data_ordered = [results_df[name].values for name in summary.index]
bp = ax.boxplot(data_ordered, labels=summary.index, patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colors[:len(names)]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Accuracy')
ax.set_title(f'Distribution of Accuracy ({N_RUNS} Runs)')
ax.tick_params(axis='x', labelsize=9)
plt.tight_layout()
plt.savefig('results_boxplot.png', dpi=150)
plt.show()
print('Saved: results_boxplot.png')

## 5. Saving Results

In [ ]:
results_df.to_csv('results_all_runs.csv', index_label='run')
summary.to_csv('results_summary.csv')
print('Saved: results_all_runs.csv, results_summary.csv')
print()
print(results_df.to_string(float_format='{:.4f}'.format))